# ADC Example 4 — External Trigger via TIM3 TRGO

**Target:** STM32F103C8T6 · ADC1 · Channel 5 (PA5) · TIM3
**Focus:** Hardware-timed sampling — the CPU never starts a conversion.

---
## Concept

Instead of calling `HAL_ADC_Start()` from software, the ADC waits for a **hardware trigger**. When TIM3 overflows, its **TRGO** (Trigger Output) pulses, and the ADC starts one conversion (or one scan sequence). This eliminates CPU jitter and gives exact, repeatable sample intervals.

---
## CubeMX Configuration

### TIM3 — Trigger Generator
| Parameter | Setting |
| :--- | :--- |
| `Clock Source` | Internal Clock (72 MHz) |
| `Prescaler` | 7200 − 1 → 10 kHz counter clock |
| `Counter Period (ARR)` | 1000 − 1 → **10 Hz update rate** |
| `Trigger Event Selection (TRGO)` | **Update Event** |

This produces one TRGO pulse every **100 ms** (10 Hz). The ADC will sample at exactly that rate.

### ADC1
| Parameter | Setting |
| :--- | :--- |
| `Scan Conversion Mode` | Disabled (single channel) |
| `Continuous Conversion Mode` | **Disabled** |
| `External Trigger Conversion Source` | **Timer 3 Trigger Out event** |
| `External Trigger Conversion Edge` | Rising edge |
| Rank 1 | Channel 5, Sampling Time 239.5 Cycles |

> The **239.5-cycle sampling time** is often used with high-impedance sources; it also ensures the conversion finishes well before the next 100 ms trigger.

### DMA
- Add **ADC1** → DMA1 Channel 1.
- **Mode:** `Circular` (continuous timed sampling) or `Normal` (fixed number of samples).
- **Data Width:** Half Word.

## Generated Initialization (key excerpt)

```c
// ADC
hadc1.Init.ContinuousConvMode  = DISABLE;
hadc1.Init.ExternalTrigConv    = ADC_EXTERNALTRIGCONV_T3_TRGO;
hadc1.Init.ExternalTrigConvEdge = ADC_EXTERNALTRIGCONVEDGE_RISING;
hadc1.Init.NbrOfConversion     = 1;

// TIM3
htim3.Init.Prescaler = 7200 - 1;
htim3.Init.Period    = 1000 - 1;
```

The HAL writes `ADC_EXTERNALTRIGCONV_T3_TRGO` into `ADC1->CR2` (`EXTSEL` bits) and enables the trigger via `EXTTRIG`.

## User Code

Start the timer **first**, then start the ADC in DMA mode. The ADC will wait for TRGO pulses.

In [ ]:
#define BUF_LEN  100
uint16_t adc_timed[BUF_LEN];

// In main(), before while(1):
HAL_TIM_Base_Start(&htim3);            // start TRGO generation
HAL_ADCEx_Calibration_Start(&hadc1);
HAL_ADC_Start_DMA(&hadc1, (uint32_t *)adc_timed, BUF_LEN);

// DMA fills adc_timed[] at exactly 10 Hz.
// After 100 samples (10 s), the DMA completion callback fires.

## DMA Completion Callback

In Circular mode, the callback fires every time the buffer is half-full and full. In Normal mode, it fires once after `BUF_LEN` samples.

In [ ]:
void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc) {
    if (hadc->Instance == ADC1) {
        // adc_timed[0..BUF_LEN-1] contain BUF_LEN samples
        // spaced exactly 100 ms apart (10 Hz)
    }
}

## Trigger Frequency Calculation

$$
f_{\text{TRGO}} = \frac{f_{\text{TIM3}}}{(\text{Prescaler}+1) \times (\text{ARR}+1)}
= \frac{72\,\text{MHz}}{7200 \times 1000} = 10\,\text{Hz}
$$

Change `ARR` to adjust the sample rate without touching the ADC configuration.

## Notes

- The ADC **must** have `ContinuousConvMode = DISABLE`; otherwise it would ignore the trigger after the first conversion.
- If you need multi-channel timed sampling, enable Scan Mode and configure the DMA buffer to hold one full sequence per trigger.
- A timer-triggered ADC is the standard building block for **FFT analysis**, **motor current sensing**, and any application where sample jitter degrades results.